In [1]:
import pandas as pd
import os

In [2]:
# Updated absolute directory and file paths
cases_path = r"C:\\Users\\bedre\\Desktop\\cdac_project\\data\\csv\\cases\\cases_2010.csv"
keys_dir = r"C:\\Users\\bedre\\Desktop\\cdac_project\\data\\csv\\keys"
base_csv_dir = r"C:\\Users\\bedre\\Desktop\\cdac_project\\data\\csv"  # Location for acts and judges

# Complete index of all metadata key dimensions
key_files = {
    "State Key": "cases_state_key.csv",
    "District Key": "cases_district_key.csv",
    "Court Key": "cases_court_key.csv",
    "Disposition Key": "disp_name_key.csv",
    "Type Name Key": "type_name_key.csv",
    "Purpose Name Key": "purpose_name_key.csv",
    "Act Key": "act_key.csv",
    "Section Key": "section_key.csv",
    "Judge Case Merge Key": "judge_case_merge_key.csv"
}

# Structural extension tables
extra_files = {
    "Acts & Sections": os.path.join(base_csv_dir, "acts_sections.csv"),
    "Judges Clean": os.path.join(base_csv_dir, "judges_clean.csv")
}

print("==================================================================")
print("🎯 CORE FACT LAYER: CASE RECORDS (2010)")
print("==================================================================")
if os.path.exists(cases_path):
    df_case = pd.read_csv(cases_path, nrows=1)
    print(f"📋 Columns: {df_case.columns.tolist()}")
    print(f"📊 Total Records: {len(pd.read_csv(cases_path))}")
    print(f"No of columns : {len(df_case.columns)}")
    print(f"💡 Sample Row: {df_case.iloc[0].to_dict()}\n")
else:
    print(f"⚠️ File not found: {cases_path}\n")

print("==================================================================")
print("⚖️ STRUCTURAL EXTENSIONS (ACTS & JUDGES)")
print("==================================================================")
for label, full_path in extra_files.items():
    print(f"🗃️ {label}")
    if os.path.exists(full_path):
        try:
            df_extra = pd.read_csv(full_path, nrows=1)
            print(f"  🔹 Columns: {df_extra.columns.tolist()}")
            print(f"  💡 Sample Row: {df_extra.iloc[0].to_dict()}")
        except Exception as e:
            print(f"  ❌ Error reading file: {e}")
    else:
        print(f"  ⚠️ File not found at: {full_path}")
    print("-" * 66)

print("\n==================================================================")
print("🗺️ DIMENSIONAL LAYER: REFERENCE KEYS")
print("==================================================================")
for label, filename in key_files.items():
    full_path = os.path.join(keys_dir, filename)
    print(f"🗂️ {label} ({filename})")
    if os.path.exists(full_path):
        try:
            df_key = pd.read_csv(full_path, nrows=1)
            print(f"  🔹 Columns: {df_key.columns.tolist()}")
            print(f"  💡 Sample Row: {df_key.iloc[0].to_dict()}")
        except Exception as e:
            print(f"  ❌ Error reading file: {e}")
    else:
        print(f"  ⚠️ File not found at: {full_path}")
    print("-" * 66)

🎯 CORE FACT LAYER: CASE RECORDS (2010)
📋 Columns: ['ddl_case_id', 'year', 'state_code', 'dist_code', 'court_no', 'cino', 'judge_position', 'female_defendant', 'female_petitioner', 'female_adv_def', 'female_adv_pet', 'type_name', 'purpose_name', 'disp_name', 'date_of_filing', 'date_of_decision', 'date_first_list', 'date_last_list', 'date_next_list']
📊 Total Records: 4281327
No of columns : 19
💡 Sample Row: {'ddl_case_id': '01-01-01-200308002162010', 'year': 2010, 'state_code': 1, 'dist_code': 1, 'court_no': 1, 'cino': 'MHNB030013812010', 'judge_position': 'chief judicial magistrate', 'female_defendant': '0 male', 'female_petitioner': '1 female', 'female_adv_def': 0, 'female_adv_pet': -9998, 'type_name': 790, 'purpose_name': 5228, 'disp_name': 42, 'date_of_filing': '2010-12-13', 'date_of_decision': '2011-06-19', 'date_first_list': '2011-06-08', 'date_last_list': '2011-06-20', 'date_next_list': '2011-06-24'}

⚖️ STRUCTURAL EXTENSIONS (ACTS & JUDGES)
🗃️ Acts & Sections
  🔹 Columns: ['ddl_c

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, datediff, to_date

# Initialize Spark Session if not already active
spark = SparkSession.builder \
    .appName("DDL-Data-Cleaning-Lab") \
    .getOrCreate()

# Load raw 2010 case file
raw_cases_path = r"C:\Users\bedre\Desktop\cdac_project\data\csv\cases\cases_2010.csv"
print("📖 Ingesting raw dataset for diagnostic cleaning...")
df_raw = spark.read.csv(raw_cases_path, header=True, inferSchema=True)

# Convert string dates to actual PySpark Date Types for accurate processing
df_with_dates = df_raw \
    .withColumn("filing_date_parsed", to_date(col("date_of_filing"), "yyyy-MM-dd")) \
    .withColumn("decision_date_parsed", to_date(col("date_of_decision"), "yyyy-MM-dd"))

# ==========================================
# CLEANING TRACK 1: RESOLVED HISTORICAL POOL
# ==========================================
cleaned_resolved_df = df_with_dates \
    .filter(col("decision_date_parsed").isNotNull() & col("filing_date_parsed").isNotNull()) \
    .withColumn("duration_days", datediff(col("decision_date_parsed"), col("filing_date_parsed"))) \
    .filter(col("duration_days") >= 0) \
    .filter((col("state_code") > 0) & (col("disp_name") > 0)) # Drop corrupt missing code markers

# ==========================================
# CLEANING TRACK 2: PENDING STREAMING POOL
# ==========================================
cleaned_pending_df = df_with_dates \
    .filter(col("date_of_decision").isNull()) \
    .filter(col("filing_date_parsed").isNotNull()) \
    .filter(col("state_code") > 0)

# Print execution diagnostics
print("\n==================================================================")
print("🧼 DATA CLEANING PROFILE SUMMARY")
print("==================================================================")
print(f"📦 Total Rows Raw Dataset   : {df_raw.count():,}")
print(f"✅ Pristine Resolved Rows   : {cleaned_resolved_df.count():,}")
print(f"⏳ Pristine Pending Rows    : {cleaned_pending_df.count():,}")
print("==================================================================")

print("\n💡 Top 3 Rows of Cleaned Resolved Data:")
cleaned_resolved_df.select("ddl_case_id", "filing_date_parsed", "decision_date_parsed", "duration_days").show(3)

Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: org.apache.spark.SparkException: Invalid Spark URL: spark://HeartbeatReceiver@Tanuja_Bedre:57288
	at org.apache.spark.rpc.RpcEndpointAddress$.apply(RpcEndpointAddress.scala:66)
	at org.apache.spark.rpc.netty.NettyRpcEnv.asyncSetupEndpointRefByURI(NettyRpcEnv.scala:143)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.executor.Executor.<init>(Executor.scala:489)
	at org.apache.spark.scheduler.local.LocalEndpoint.<init>(LocalSchedulerBackend.scala:66)
	at org.apache.spark.scheduler.local.LocalSchedulerBackend.start(LocalSchedulerBackend.scala:136)
	at org.apache.spark.scheduler.TaskSchedulerImpl.start(TaskSchedulerImpl.scala:229)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:629)
	at org.ap